# Egyptian Hieroglyphic NMT — Comprehensive Cleaning Pipeline

**السكريبت ده بيحمل الداتا مباشرة من HuggingFace ومحتاجش أي CSV input منك.**

الـ datasets:
- `thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium`
- `phiwi/bbaw_egyptian`

**الـ pipeline بيعمل (بالترتيب):**
1. تحميل الـ 2 datasets من HuggingFace + دمجهم
2. تطبيق **Table 4 filter** (POS=`/` + lKey مفقود → drop)
3. تنظيف الترانسليتيريشن (PDF Table 2)
4. تنظيف الترجمة الألمانية (PDF Table 1)
5. حذف الصفوف الفارغة بعد التنظيف
6. **حذف المتكرر** (full duplicates على الزوج المنظف) + إحصاء كم اتشال
7. **كشف اللغة** على عمود `clean_german` + إحصاء (German/English/Other/Unknown)
8. **حذف English + Unknown** + إحصاء كم اتشال
9. حساب **Confidence score** لكل صف (data quality 0→1)
10. حفظ:
    - `dataset_cleaned_final.csv` — للتدريب الأساسي (4 أعمدة + language + confidence)
    - `dataset_low_conf.csv` — صفوف مشكوك فيها (للمراجعة)
    - `character_level.csv` — للـ ByT5 segmentation
    - `cleaning_stats.txt` — تقرير JSON بكل الأرقام

**Configuration واحدة بس مهمة قبل ما تعمل Run All:**
- `REMOVE_GERMAN_PUNCTUATION` (افتراضي `False`) — احتفظ بـ `.` `?` `!`. غيرها لـ `True` بس لو مصمم.
- `MIN_CONF_THRESHOLD` (افتراضي `0.5`) — كل اللي تحتها بيروح للـ low_conf file.

**المصادر:**
- Wiesenbach & Riezler (2019), *Multi-Task Modeling of Phonographic Languages*, IWSLT
- Höper et al. (2018), *Berlin Text System 3.1 User Manual*
- van den Berg, *Manuel de Codage*
- `cleansing_operations.pdf` — Tables 1, 2, 3, 4


## 1 — Install Dependencies

In [ ]:
# Run this once. On Kaggle/Colab, internet must be ON for HF download.
!pip install -q langdetect datasets pandas
print("Dependencies installed ✅")


## 2 — Configuration

الـ knobs الوحيدة اللي محتاج تعدل فيها (لو محتاج):

In [ ]:
# =============================================================================
#  CONFIGURATION
# =============================================================================

# علامات الترقيم في الألماني (.  ?  !)
REMOVE_GERMAN_PUNCTUATION = False    # افتراضي: احتفظ بيها (أحسن للترجمة)

# فلترة الطول (لاستبعاد الجمل الطويلة جداً اللي عادة noisy)
MIN_TRANSLIT_TOKENS = 1
MAX_TRANSLIT_TOKENS = 150
MIN_GERMAN_TOKENS   = 1
MAX_GERMAN_TOKENS   = 150

# نسبة طول الترانسليتيريشن للألماني (alignment sanity check)
MAX_TOKEN_RATIO = 5.0

# عتبة الـ confidence — اللي تحتها يروح للـ low_conf file
MIN_CONF_THRESHOLD = 0.5

# Language detection
DROP_OTHER_LANGUAGES = False       # False = خلي اللاتيني/الفرنساوي (ممكن يكون صحيح)
                                    # True  = شيله مع الإنجليزي والـ unknown
LANGDETECT_MIN_LEN  = 5
LANGDETECT_MIN_PROB = 0.85

# مكان الـ output. على Kaggle/Colab "." شغالة
OUTPUT_DIR = '.'

SEED = 42


## 3 — Imports & Setup

In [ ]:
import os
import re
import json
import random
import unicodedata

import pandas as pd
import numpy as np

from datasets import load_dataset
from langdetect import detect_langs, DetectorFactory, LangDetectException

# Make langdetect deterministic
DetectorFactory.seed = SEED
random.seed(SEED)
np.random.seed(SEED)

print("Setup complete ✅")


## 4 — Transliteration Cleaner (PDF Table 2)

كل قواعد PDF Table 2 + إضافات:
- **Unicode NFC normalization** (يوحد combining chars)
- إزالة brackets مع الإبقاء على المحتوى
- `=` clitic splitting (`n=f` → `n f`)
- hyphen → space (للـ alignment مع الألماني)
- إزالة noise chars (`!ø⁝:+~`)
- إزالة grammatical tags (`.PL` `.DU`)
- Drop sentences fully made of destruction markers


In [ ]:
# =============================================================================
#  TRANSLITERATION CLEANER (PDF TABLE 2)
# =============================================================================

RE_GRAM_TAG_PAREN     = re.compile(r'\(\s*\.[A-Z]{2,}[A-Za-z0-9]*\s*\)')
RE_DOT_UPPERCASE_TAG  = re.compile(r'\.[A-Z]{2,}[A-Za-z0-9]*')
RE_PL_MARKER          = re.compile(r'\.pl\b|\.Pl\b|,pl\b|\bpl\b|\{\.?pl\}|\.\{pl\}')
RE_DU_MARKER          = re.compile(r'\.du\b|,du\b')
RE_SIMPLE_NOISE       = re.compile(r'\b(ON|GN|Pr[äa]p\.?|oder\s+ḫr\s*=\s*s|L[üu]ckeL[üu]ckeGap)\b')
RE_TRANS_NOISE_CHARS  = re.compile(r'[!ø⁝:+~]')
RE_SQUARE_DESTRUCTION = re.compile(r'\.\.\s*\d+\s*Q\s*\.\.')
RE_TRANS_BRACKETS     = re.compile(r'[(){}\[\]⸢⸣⟨⟩〈〉𓍹𓍺𓊆𓊇𓉘𓊂]')
RE_TRANS_PUNCT        = re.compile(r'[?⸮.,]')
RE_TRANS_HYPHEN       = re.compile(r'[-‑‐‒–—]')
RE_TRANS_UNDERSCORE   = re.compile(r'[_⸗]')

TRANS_KILL_PATTERNS = [
    re.compile(r'^\s*[-\.…_?⸮]+\s*$'),
    re.compile(r'^\s*-\?\?-\s*$'),
    re.compile(r'^\s*\.\s*\.\s*\.\s*$'),
    re.compile(r'^\s*/+\s*$'),
]


def _split_equals(text: str) -> str:
    out = []
    for tok in text.split():
        parts = [p for p in tok.split('=') if p]
        out.extend(parts)
    return ' '.join(out)


def clean_transliteration(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''

    t = unicodedata.normalize('NFC', text)

    for pat in TRANS_KILL_PATTERNS:
        if pat.match(t):
            return ''

    t = t.replace('Ꜥ', 'ꜥ')
    t = t.replace('ʿ', 'ꜥ')
    t = t.replace('≡', '=')
    t = RE_SQUARE_DESTRUCTION.sub(' / ', t)
    t = RE_GRAM_TAG_PAREN.sub('', t)
    t = RE_TRANS_BRACKETS.sub('', t)
    t = RE_DOT_UPPERCASE_TAG.sub('', t)
    t = RE_PL_MARKER.sub('', t)
    t = RE_DU_MARKER.sub('', t)
    t = RE_SIMPLE_NOISE.sub('', t)
    t = RE_TRANS_PUNCT.sub('', t)
    t = RE_TRANS_HYPHEN.sub(' ', t)
    t = _split_equals(t)
    t = RE_TRANS_UNDERSCORE.sub(' ', t)
    t = RE_TRANS_NOISE_CHARS.sub('', t)
    return ' '.join(t.split())


# Smoke test
_t_tests = [
    ('md(w).t',          'mdwt'),
    ('kꜣ(.PL)',          'kꜣ'),
    ('n=f',              'n f'),
    ('=sn',              'sn'),
    ('⸢nb⸣',             'nb'),
    ('ḫrp-ḥr(.ꞽ)-ꞽb',    'ḫrp ḥrꞽ ꞽb'),
    ('Ꜥḥꜥ',              'ꜥḥꜥ'),
    ('m ..2Q.. ḥr',      'm / ḥr'),
    ('//',               ''),
]
print("Transliteration cleaner smoke test:")
for raw, want in _t_tests:
    got = clean_transliteration(raw)
    print(f"  {'✅' if got == want else '❌'}  {raw!r:25s} → {got!r}")


## 5 — German Cleaner (PDF Table 1) — الأقوى

كل قواعد PDF Table 1 + إضافات للترجمة الـ NMT:
- حفظ الـ hyphens في أسماء العلم (`Nemti-em-za-ef`)
- حذف brackets-with-content (commentary, glosses)
- destruction markers → drop the whole sentence
- `LHG/LPH` → `Leben, Heil, Gesundheit`
- `OÄ/UÄ` → `Oberägypten/Unterägypten`
- `Brot/Bier` → `Brot` (slash pairs)
- editorial marks (pilcrow, guillemets, daggers) → remove
- `Halt!!!` → `Halt!` (repeated punctuation)
- `Wirklich...` → `Wirklich` (trailing ellipsis)
- Unicode NFC normalisation


In [ ]:
# =============================================================================
#  GERMAN CLEANER (PDF TABLE 1)
# =============================================================================

DE_FULL_KILL_IF_ALONE = [
    r'^\s*\?+\s*$',
    r'^\s*[-\.…_⸮\?\s]+$',
    r'^\s*\.\s*\.\s*\.\s*$',
    r'^\s*⸮\s*_\s*\?\s*$',
    r'^\s*-+\s*\?+\s*-+\s*$',
    r'^\s*\[?---\]?\s*$',
    r'^\s*〈\s*〉\s*-+\s*$',
]
RE_DE_FULL_KILL_IF_ALONE = [re.compile(p) for p in DE_FULL_KILL_IF_ALONE]

DE_DESTRUCTION_MARKERS = [
    r'--\s*zerst[öo]rt\s*--',
    r'--\s*Zerst[öo]rung\s*--',
    r'--\s*Zeichenreste\s*--',
    r'--\s*Beischrift\s+zerst[öo]rt\s*--',
    r'--\s*unklar\s*--',
    r'keine\s+Übersetzung\s+vorhanden',
    r'Keine\s+Übersetzung\s+möglich',
    r'---\s*LEER\s+GEFUNDEN\s*---',
    r'No\s+translation\s+available',
    r'No\s+translation\s+possible',
    r'---\s*FOUND\s+EMPTY\s*---',
    r'-\s*Variante[^-]*-',
]
RE_DE_DESTRUCTION = [re.compile(p, re.IGNORECASE) for p in DE_DESTRUCTION_MARKERS]

DE_DELETE_WITH_CONTENT = [
    r'\[§[^\]]*\]',
    r'§\s*[\d]+[a-zA-Z\-]*[\s.,:]?',
    r'\$\[[^\]]*\]\$',
    r'\(\s*wört[^)]*\)',
    r'\(\s*wört[^)]*$',
    r'\[\s*ältere\s+Fassung[^\]]*\]',
    r'\(\s*älterer\s+Text[^)]*\)',
    r'\(\s*oder[^)]*\)',
    r'\[[^\]]*Beischrift[^\]]*\]\s*:?',
    r'\(\s*d\.\s*h\.?[^)]*\)',
    r'\(\s*i\.\s*e\.?[^)]*\)',
    r'\(\s*\?\s*\)',
    r'\(\s*=\s[^)]*\)',
    r'\(\s*Glosse[^)]*\)',
]
RE_DE_DELETE_WITH_CONTENT = [re.compile(p, re.IGNORECASE) for p in DE_DELETE_WITH_CONTENT]

DE_LHG_PATTERNS = [
    (re.compile(r'-\s*\{?\s*LHG\s*\}?\s*LHG\s*-'), 'Leben, Heil, Gesundheit'),
    (re.compile(r'-\s*LHG\s*-'),                    'Leben, Heil, Gesundheit'),
    (re.compile(r'-\s*LHG\b'),                       'Leben, Heil, Gesundheit'),
    (re.compile(r'\bLHG\b'),                         'Leben, Heil, Gesundheit'),
    (re.compile(r'\bLPH\b'),                         'Leben, Heil, Gesundheit'),
    (re.compile(r'\bl\.h[\.,\-]?g[\.\-\s]*'),        'Leben, Heil, Gesundheit'),
    (re.compile(r'\bl\.p\.h\.?'),                    'Leben, Heil, Gesundheit'),
]

DE_EGYPT_PATTERNS = [
    (re.compile(r'\bO\.?\s*Äg\.?'),     'Oberägypten'),
    (re.compile(r'\bO\.?\s*Ä\.?(?!g)'), 'Oberägypten'),
    (re.compile(r'\bU\.?\s*Äg\.?'),     'Unterägypten'),
    (re.compile(r'\bU\.?\s*Ä\.?(?!g)'), 'Unterägypten'),
]

DE_FRENCH_PATTERNS = [
    (re.compile(r'"arbustes\s+à\s+épines"'), 'dornige Sträucher'),
    (re.compile(r'\brôdeurs\b'),              'Plünderer'),
]

RE_DE_NN_PATTERNS = [re.compile(r'--NN--'), re.compile(r'\|NN\|'), re.compile(r'\bNN\b')]
RE_DE_STRAY_PIPE  = re.compile(r'\|')
RE_DE_DOUBLE_TRANS_CURLY_FIRST  = re.compile(r'\{([^{}]*)\}\s*〈[^〉]*〉')
RE_DE_DOUBLE_TRANS_CURLY_SECOND = re.compile(r'〈[^〉]*〉\s*\{([^{}]*)\}')
RE_DE_SLASH_PAIR        = re.compile(r'([^\s/(){}\[\].,;:!?]+)\s*/\s*([^\s/(){}\[\].,;:!?]+)')
RE_DE_VARIANTE_PREFIX   = re.compile(r'^\s*\(?\s*Variante\s*[:\)]?\s*', re.IGNORECASE)
RE_DE_PAREN_INNER       = re.compile(r'\([^()]*\)')
RE_DE_REMAINING_BRACKETS = re.compile(r'[{}\[\]⟨⟩〈〉⸢⸣<>𓉘𓊂𓍹𓍺"„"‚‘$#]')
RE_DE_EDITORIAL_MARKS    = re.compile(r'[❡¶†‡»«›‹❬❭❮❯]')
RE_DE_HYPHEN_LINEBREAK   = re.compile(r'-\s*[\r\n]+\s*')
RE_DE_SPACE_BEFORE_PUNCT = re.compile(r'\s+([.,;:!?])')
RE_DE_TRAILING_ELLIPSIS  = re.compile(r'\s*\.{3,}\s*$')
RE_DE_LEADING_JUNK       = re.compile(r'^[\s\.\-–—\'"‚‘„"]+')
RE_DE_TRAILING_JUNK      = re.compile(r'[\s\-–—\'"‚‘„"]+$')
RE_DE_REPEATED_PUNCT     = re.compile(r'([.!?,;:])\1{1,}')


def clean_german(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text.strip()
    if t in ('', 'None', 'nan', 'NaN', 'none'):
        return ''

    t = unicodedata.normalize('NFC', t)

    t = RE_DE_HYPHEN_LINEBREAK.sub('-', t)
    t = t.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
    t = ' '.join(t.split())

    t = t.replace('Ꜥ', 'ꜥ')
    t = t.replace('`', "'")
    t = t.replace('≡', '=')
    t = t.replace('&', ' und ')
    t = RE_DE_EDITORIAL_MARKS.sub('', t)

    for rx in RE_DE_FULL_KILL_IF_ALONE:
        if rx.match(t):
            return ''

    for rx in RE_DE_DESTRUCTION:
        if rx.search(t):
            return ''

    for rx, repl in DE_LHG_PATTERNS:
        t = rx.sub(repl, t)
    for rx, repl in DE_EGYPT_PATTERNS:
        t = rx.sub(repl, t)
    for rx, repl in DE_FRENCH_PATTERNS:
        t = rx.sub(repl, t)

    for rx in RE_DE_DELETE_WITH_CONTENT:
        t = rx.sub('', t)

    for rx in RE_DE_NN_PATTERNS:
        t = rx.sub('', t)
    t = RE_DE_STRAY_PIPE.sub('', t)

    t = RE_DE_VARIANTE_PREFIX.sub('', t)

    t = RE_DE_DOUBLE_TRANS_CURLY_FIRST.sub(lambda m: m.group(1), t)
    t = RE_DE_DOUBLE_TRANS_CURLY_SECOND.sub(lambda m: m.group(1), t)

    prev = None
    while prev != t:
        prev = t
        t = RE_DE_SLASH_PAIR.sub(r'\1', t)

    prev = None
    while prev != t:
        prev = t
        t = RE_DE_PAREN_INNER.sub('', t)

    t = RE_DE_REMAINING_BRACKETS.sub('', t)
    t = RE_DE_TRAILING_ELLIPSIS.sub('', t)
    t = RE_DE_LEADING_JUNK.sub('', t)
    t = RE_DE_TRAILING_JUNK.sub('', t)
    t = RE_DE_SPACE_BEFORE_PUNCT.sub(r'\1', t)
    t = RE_DE_REPEATED_PUNCT.sub(r'\1', t)
    t = ' '.join(t.split())

    if REMOVE_GERMAN_PUNCTUATION:
        t = re.sub(r'[.?!,;:]', '', t)
        t = ' '.join(t.split())

    if not t or re.fullmatch(r'[\s\.,;:\-!?]+', t):
        return ''

    return t


# Smoke test
_d_tests = [
    ('--zerstört--',                              ''),
    ('keine Übersetzung vorhanden',               ''),
    ('(es) werde zerrieben.',                     'werde zerrieben.'),
    ('Nemti-em-za-ef ist hier.',                  'Nemti-em-za-ef ist hier.'),
    ('König von OÄ und UÄ',                       'König von Oberägypten und Unterägypten'),
    ('Pharao - LHG - ist da',                     'Pharao Leben, Heil, Gesundheit ist da'),
    ('Brot/Bier',                                 'Brot'),
    ('Tag & Nacht',                               'Tag und Nacht'),
    ('Was ist es?',                               'Was ist es?'),
    ('Halt!!!',                                   'Halt!'),
    ('Wirklich...',                               'Wirklich'),
]
print("German cleaner smoke test:")
for raw, want in _d_tests:
    got = clean_german(raw)
    print(f"  {'✅' if got == want else '❌'}  {raw!r:48s} → {got!r}")


## 6 — Per-row Confidence Score

كل صف بياخد score من 0 لـ 1 على حسب جودة البيانات. الـ low-conf rows بتروح لملف منفصل (مش بتترمى).

**العقوبات:**
- نسبة cleaning loss عالية (راح أكتر من 70% من الخام)
- token alignment سيء (الترانسليتيريشن أطول من الألماني بكتير أو العكس)
- artifacts متبقية (`(` `[` إلخ)
- letter density قليلة في الألماني (يدل على noise)
- طول الجملة برة الـ range


In [ ]:
# =============================================================================
#  CONFIDENCE SCORE
# =============================================================================

ARTIFACT_CHARS = set('()[]{}〈〉⸢⸣⟨⟩<>𓉘𓊂𓍹𓍺')


def compute_confidence(row) -> float:
    raw_t   = row.get('raw_transliteration', '') or ''
    clean_t = row.get('clean_transliteration', '') or ''
    raw_g   = row.get('raw_german', '') or ''
    clean_g = row.get('clean_german', '') or ''

    if not clean_t.strip() or not clean_g.strip():
        return 0.0

    score = 1.0

    # cleaning-loss penalty
    if len(raw_g) > 0:
        keep = len(clean_g) / len(raw_g)
        if   keep < 0.30: score -= 0.20
        elif keep < 0.50: score -= 0.10

    # length penalty
    t_tok = clean_t.split()
    g_tok = clean_g.split()
    if len(t_tok) < MIN_TRANSLIT_TOKENS or len(g_tok) < MIN_GERMAN_TOKENS:
        score -= 0.30
    if len(t_tok) > MAX_TRANSLIT_TOKENS or len(g_tok) > MAX_GERMAN_TOKENS:
        score -= 0.20

    # token-ratio alignment
    if t_tok and g_tok:
        ratio = max(len(t_tok), len(g_tok)) / min(len(t_tok), len(g_tok))
        if ratio > MAX_TOKEN_RATIO:
            score -= 0.20

    # artifact characters left in German
    if any(c in clean_g for c in ARTIFACT_CHARS):
        score -= 0.30

    # letter density
    g_letters = sum(1 for c in clean_g if c.isalpha())
    if len(clean_g) > 0:
        density = g_letters / len(clean_g)
        if density < 0.40:
            score -= 0.20

    # bonus for healthy length
    if 4 <= len(g_tok) <= 30:
        score += 0.05

    return float(max(0.0, min(1.0, score)))


# Demo
_demo = pd.DataFrame([
    {'raw_transliteration': 'nḏ wdi̯ r =s', 'clean_transliteration': 'nḏ wdi̯ r s',
     'raw_german': '(es) werde zerrieben.', 'clean_german': 'werde zerrieben.'},
    {'raw_transliteration': '...', 'clean_transliteration': '',
     'raw_german': '--zerstört--',  'clean_german': ''},
])
_demo['conf'] = _demo.apply(compute_confidence, axis=1)
print("Confidence demo:")
print(_demo[['clean_transliteration', 'clean_german', 'conf']].to_string())


## 7 — Language Detection (Robust)

`langdetect` بشكل عشوائي افتراضياً. هنا:
- `DetectorFactory.seed = 42` → deterministic
- نستخدم `detect_langs` (probabilities) مع threshold عشان نتجنب الـ misclassification
- نصوص قصيرة جداً (< 5 حروف) تعتبر `unknown` مباشرة

النتيجة: `german` / `english` / `other` / `unknown` / `uncertain`


In [ ]:
# =============================================================================
#  LANGUAGE DETECTION
# =============================================================================

def detect_language(text: str) -> str:
    if not isinstance(text, str):
        return 'unknown'
    s = text.strip()
    if len(s) < LANGDETECT_MIN_LEN:
        return 'unknown'
    try:
        results = detect_langs(s)
        if not results:
            return 'unknown'
        top = results[0]
        if top.prob < LANGDETECT_MIN_PROB:
            return 'uncertain'
        if top.lang == 'de':
            return 'german'
        if top.lang == 'en':
            return 'english'
        return 'other'
    except LangDetectException:
        return 'unknown'


# Tests
_l_tests = [
    ("Der Pharao ist mächtig.",                 'german'),
    ("The pharaoh is mighty.",                  'english'),
    ("Mille pains et mille jarres de bière.",   'other'),
    ("..",                                      'unknown'),
]
print("Language detection test:")
for txt, want in _l_tests:
    got = detect_language(txt)
    mark = '✅' if got == want else 'ℹ️'
    print(f"  {mark}  {txt[:40]!r:42s} → {got}  (expected {want})")


## 8 — Load Both HuggingFace Datasets

دلوقتي بيتم التحميل المباشر من HF — مش محتاج أي input منك. كل اللي محتاج: internet.


In [ ]:
# =============================================================================
#  HF DATASET LOADER + COLUMN DETECTION
# =============================================================================

_TRANS_COL_PRIORITY = [
    'transliteration', 'translit', 'transcription', 'text',
    'lemma', 'form', 'word', 'token', 'raw', 'original', 'value',
]
_GERMAN_COL_PRIORITY = [
    'german_translation', 'translation_de', 'translation',
    'de', 'german', 'übersetzung', 'uebersetzung',
    'meaning', 'gloss', 'gloss_de', 'sense',
]
_POS_COL_PRIORITY  = ['pos', 'part_of_speech', 'wordclass', 'class']
_LKEY_COL_PRIORITY = ['lkey', 'lemma_id', 'lemma_key', 'wkey']


def _pick(cols, priority):
    lower = {c.lower(): c for c in cols}
    for p in priority:
        if p in lower:
            return lower[p]
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in ('transl', 'german', 'übersetz')):
            return c
    return None


def detect_columns(cols):
    return {
        'translit': _pick(cols, _TRANS_COL_PRIORITY),
        'german'  : _pick(cols, _GERMAN_COL_PRIORITY),
        'pos'     : _pick(cols, _POS_COL_PRIORITY),
        'lkey'    : _pick(cols, _LKEY_COL_PRIORITY),
    }


def _flatten(s):
    return ' '.join(s.split()) if isinstance(s, str) else ''


def load_hf(dataset_id: str) -> pd.DataFrame:
    print(f"\n→ Loading {dataset_id}")
    try:
        ds = load_dataset(dataset_id, split='train')
    except Exception as e:
        print(f"  WARN train split failed → trying first split  ({e})")
        d = load_dataset(dataset_id)
        ds = d[list(d.keys())[0]]
    df = ds.to_pandas()
    cols = detect_columns(list(df.columns))
    print(f"  rows: {len(df):,}   detected cols: {cols}")

    out = pd.DataFrame()
    out['raw_transliteration'] = (
        df[cols['translit']].astype(str).apply(_flatten) if cols['translit'] else ''
    )
    out['raw_german'] = (
        df[cols['german']].astype(str).apply(_flatten)
        if cols['german'] else pd.Series([''] * len(df))
    )
    if cols['pos']:
        out['_pos'] = df[cols['pos']].astype(str)
    if cols['lkey']:
        out['_lkey'] = df[cols['lkey']].astype(str)
    out['source'] = dataset_id.split('/')[-1]
    return out


# Load both
df1 = load_hf('thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium')
df2 = load_hf('phiwi/bbaw_egyptian')

merged = pd.concat([df1, df2], ignore_index=True)
print(f"\n✅ Merged total rows: {len(merged):,}")
print(f"   columns: {list(merged.columns)}")


## 9 — Run Full Pipeline

دلوقتي بنطبق كل الخطوات بالترتيب على الداتا اللي اتحملت:

1. Table 4 filter
2. تنظيف
3. حذف الفارغ
4. حذف المتكرر (مع إحصاء)
5. كشف اللغة (مع إحصاء)
6. حذف English/Unknown (مع إحصاء)
7. Confidence scores
8. حفظ كل الـ outputs


In [ ]:
# =============================================================================
#  FULL PIPELINE
# =============================================================================

stats = {}
df = merged.copy()
stats['rows_initial'] = len(df)
print("=" * 72)
print(f"  STAGE 0: Loaded merged dataset → {len(df):,} rows")
print("=" * 72)

# ---- 1. TABLE 4 FILTER ----------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 1: PDF TABLE 4 FILTER  (POS='/' + lKey missing → drop)")
print("=" * 72)
if '_pos' in df.columns:
    if '_lkey' in df.columns:
        mask = (df['_pos'].str.strip() == '/') & (
            df['_lkey'].isna() |
            (df['_lkey'].str.strip() == '') |
            (df['_lkey'].str.lower() == 'nan')
        )
    else:
        mask = (df['_pos'].str.strip() == '/')
    n = int(mask.sum())
    df = df[~mask].reset_index(drop=True)
    stats['table4_dropped'] = n
    print(f"  Dropped {n:,} rows by Table 4 rule")
else:
    stats['table4_dropped'] = 0
    print("  (no POS column → skipping)")

for c in ('_pos', '_lkey'):
    if c in df.columns:
        df = df.drop(columns=[c])

# ---- 2. APPLY CLEANERS -----------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 2: APPLY CLEANERS")
print("=" * 72)
print("  cleaning transliteration ...")
df['clean_transliteration'] = df['raw_transliteration'].apply(clean_transliteration)
print("  cleaning German ...")
df['clean_german'] = df['raw_german'].apply(clean_german)
print(f"  done.  rows: {len(df):,}")

# ---- 3. DROP EMPTY ---------------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 3: DROP ROWS WITH EMPTY CLEANED COLUMNS")
print("=" * 72)
before = len(df)
df = df[
    (df['clean_transliteration'].str.len() > 0) &
    (df['clean_german'].str.len() > 0)
].reset_index(drop=True)
stats['empty_after_clean'] = int(before - len(df))
print(f"  Dropped {before - len(df):,} rows")
print(f"  Remaining: {len(df):,}")

# ---- 4. DEDUPLICATE --------------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 4: DEDUPLICATE  (on cleaned pair)")
print("=" * 72)
before = len(df)
df = df.drop_duplicates(
    subset=['clean_transliteration', 'clean_german']
).reset_index(drop=True)
stats['duplicates_dropped'] = int(before - len(df))
print(f"  Duplicate pairs dropped: {before - len(df):,}")
print(f"  Remaining: {len(df):,}")

# ---- 5. LANGUAGE DETECTION -------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 5: LANGUAGE DETECTION on clean_german")
print("=" * 72)
print("  (running langdetect — may take ~1 min on ~100K rows)")
df['language'] = df['clean_german'].apply(detect_language)

lang_counts = df['language'].value_counts().to_dict()
stats['language_counts'] = {k: int(v) for k, v in lang_counts.items()}
print("\n  Language distribution:")
for k, v in sorted(lang_counts.items(), key=lambda x: -x[1]):
    pct = 100 * v / len(df)
    print(f"    {k:12s} : {v:>7,d}  ({pct:.2f}%)")

# ---- 6. DROP ENGLISH + UNKNOWN ---------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 6: DROP ENGLISH + UNKNOWN")
print("=" * 72)
drop_set = {'english', 'unknown'}
if DROP_OTHER_LANGUAGES:
    drop_set |= {'other', 'uncertain'}
n_eng     = int((df['language'] == 'english').sum())
n_unk     = int((df['language'] == 'unknown').sum())
n_other   = int((df['language'] == 'other').sum())
n_unc     = int((df['language'] == 'uncertain').sum())
print(f"  english     : {n_eng:>7,d}  → drop")
print(f"  unknown     : {n_unk:>7,d}  → drop")
print(f"  other       : {n_other:>7,d}  → {'drop' if 'other' in drop_set else 'keep'}")
print(f"  uncertain   : {n_unc:>7,d}  → {'drop' if 'uncertain' in drop_set else 'keep'}")

before = len(df)
df = df[~df['language'].isin(drop_set)].reset_index(drop=True)
stats['english_dropped'] = n_eng
stats['unknown_dropped'] = n_unk
stats['language_total_dropped'] = int(before - len(df))
print(f"\n  Total dropped: {before - len(df):,}")
print(f"  Remaining: {len(df):,}")

# ---- 7. CONFIDENCE ---------------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 7: COMPUTE CONFIDENCE SCORES")
print("=" * 72)
df['confidence'] = df.apply(compute_confidence, axis=1)
print(f"  conf mean : {df['confidence'].mean():.3f}")
print(f"  conf std  : {df['confidence'].std():.3f}")
print("  conf distribution:")
bins   = [0.0, 0.3, 0.5, 0.7, 0.85, 1.01]
labels = ['0.0-0.3', '0.3-0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']
df['_b'] = pd.cut(df['confidence'], bins=bins, labels=labels, right=False)
for lbl, n in df['_b'].value_counts().sort_index().items():
    print(f"    {lbl:>10s}: {n:>7,d}  ({100 * n / len(df):.1f}%)")
df = df.drop(columns=['_b'])

# ---- 8. SPLIT BY CONFIDENCE -----------------------------------------------
print("\n" + "=" * 72)
print(f"  STAGE 8: SPLIT (conf >= {MIN_CONF_THRESHOLD} → main, conf < → low_conf)")
print("=" * 72)
high = df[df['confidence'] >= MIN_CONF_THRESHOLD].reset_index(drop=True)
low  = df[df['confidence'] <  MIN_CONF_THRESHOLD].reset_index(drop=True)
stats['rows_high_conf'] = len(high)
stats['rows_low_conf']  = len(low)
print(f"  high-conf : {len(high):,}  → goes to dataset_cleaned_final.csv")
print(f"  low-conf  : {len(low):,}  → goes to dataset_low_conf.csv")

# ---- 9. SAVE ---------------------------------------------------------------
print("\n" + "=" * 72)
print("  STAGE 9: SAVE OUTPUTS")
print("=" * 72)
os.makedirs(OUTPUT_DIR, exist_ok=True)

main_cols = ['raw_transliteration', 'clean_transliteration',
             'raw_german', 'clean_german', 'language', 'confidence']

p_main = os.path.join(OUTPUT_DIR, 'dataset_cleaned_final.csv')
high[main_cols].to_csv(p_main, index=False, encoding='utf-8-sig')
print(f"  ✅ {p_main}  ({len(high):,} rows × {len(main_cols)} cols)")

p_low = os.path.join(OUTPUT_DIR, 'dataset_low_conf.csv')
low[main_cols].to_csv(p_low, index=False, encoding='utf-8-sig')
print(f"  ✅ {p_low}  ({len(low):,} rows × {len(main_cols)} cols)")

# Character-level (from high-conf only — for ByT5 segmentation)
def _to_char_seq(t):
    return ' '.join(' '.join(list(w)) for w in t.split())
char_df = pd.DataFrame({
    'characters': high['clean_transliteration'].apply(_to_char_seq),
    'clean_text': high['clean_transliteration'],
}).reset_index(drop=True)
p_char = os.path.join(OUTPUT_DIR, 'character_level.csv')
char_df.to_csv(p_char, index=False, encoding='utf-8-sig')
print(f"  ✅ {p_char}  ({len(char_df):,} rows × 2 cols)")

# Stats report
stats['final_main_rows']      = len(high)
stats['final_low_conf_rows']  = len(low)
stats['final_charlevel_rows'] = len(char_df)
stats['config'] = {
    'REMOVE_GERMAN_PUNCTUATION': REMOVE_GERMAN_PUNCTUATION,
    'MIN_CONF_THRESHOLD':        MIN_CONF_THRESHOLD,
    'MAX_TOKEN_RATIO':           MAX_TOKEN_RATIO,
    'DROP_OTHER_LANGUAGES':      DROP_OTHER_LANGUAGES,
}
p_stats = os.path.join(OUTPUT_DIR, 'cleaning_stats.txt')
with open(p_stats, 'w', encoding='utf-8') as f:
    f.write("CLEANING PIPELINE — REPORT\n" + "=" * 60 + "\n\n")
    f.write(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"  ✅ {p_stats}")

print("\n" + "=" * 72)
print(f"  PIPELINE COMPLETE  →  final training rows: {len(high):,}")
print("=" * 72)


## 10 — Random Sample Inspection

In [ ]:
# Random sample of high-conf
print("=" * 72)
print("  RANDOM SAMPLE — high-confidence rows (these go to training)")
print("=" * 72)
pd.set_option('display.max_colwidth', 100)
for i, row in high.sample(min(10, len(high)), random_state=SEED).iterrows():
    t_tok = len(row['clean_transliteration'].split())
    g_tok = len(row['clean_german'].split())
    print(f"\n[row {i}]  conf={row['confidence']:.2f}  "
          f"trans_tok={t_tok}  german_tok={g_tok}  lang={row['language']}")
    print(f"  RAW   trans  : {row['raw_transliteration'][:90]}")
    print(f"  CLEAN trans  : {row['clean_transliteration'][:90]}")
    print(f"  RAW   german : {row['raw_german'][:90]}")
    print(f"  CLEAN german : {row['clean_german'][:90]}")

print("\n\n" + "=" * 72)
print("  RANDOM SAMPLE — low-confidence rows (مراجعة يدوية)")
print("=" * 72)
if len(low) > 0:
    for i, row in low.sample(min(5, len(low)), random_state=SEED).iterrows():
        print(f"\n[row {i}]  conf={row['confidence']:.2f}  lang={row['language']}")
        print(f"  CLEAN trans  : {row['clean_transliteration'][:90]}")
        print(f"  CLEAN german : {row['clean_german'][:90]}")

print("\n\n" + "=" * 72)
print("  CHARACTER-LEVEL preview (للـ ByT5 segmentation)")
print("=" * 72)
for _, row in char_df.head(5).iterrows():
    print(f"\n  chars : {row['characters'][:80]}")
    print(f"  clean : {row['clean_text'][:80]}")


## 11 — نصايح للتدريب بعد كده

### عشان تحسن الـ BLEU من 2.75:

1. **استخدم ByT5 أو mT5 للترجمة** — مش transformer من الصفر.

2. **Sequence lengths**: max=128 للـ input والـ output. أكتر = ضياع GPU.

3. **Backtranslation**: لو عندك جمل ترانسليتيريشن بدون ترجمة (الـ TLA فيها كده)، درب موديل عكسي وولّد synthetic pairs. Heidelberg group عملوها.

4. **Evaluation**: استخدم `sacrebleu` (standard tokenization)، مش nltk:
   ```python
   import sacrebleu
   bleu = sacrebleu.corpus_bleu(predictions, [references])
   chrf = sacrebleu.corpus_chrf(predictions, [references])
   ```

5. **chrF أحسن من BLEU** للغات low-resource. تشوف عندك chrF=15-30 في الصورة، ده مش وحش زي ما البليو بيقول.

6. **Beam search**: `num_beams=5` أحسن من greedy.

7. **Confidence filtering**: لو عايز تجرب حاجة قوية، درب على `conf >= 0.7` بس الأول. ده هيشيل الـ noise المخبي.

### للـ ByT5 segmentation:
- نتايجك (EM=0.66, Levenshtein 0.95+) كويسة جداً.
- جرب augmentation بسيط: عشوائياً ادمج كلمتين أو فصل واحدة، ده يدي الموديل diversity.

---

### الـ Outputs:
| File | Use |
|---|---|
| `dataset_cleaned_final.csv` | الـ translation model الأساسي |
| `character_level.csv`       | الـ ByT5 segmentation |
| `dataset_low_conf.csv`      | راجعها يدوياً، فيها ممكن صفوف كويسة |
| `cleaning_stats.txt`        | تقرير شامل بالأرقام |
